In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u

repo_root=Path(".")
os.chdir(repo_root)

if str(repo_root) not in sys.path:
    sys.path.insert(0,str(repo_root))

from scripts import compute_UVLF_cv as cv

In [ ]:
data_dir=repo_root/"data"/"For_Jiahui_Aug17"

base_sfr=np.loadtxt(data_dir/"basekenn_VSMDPL_z10_mhalo_SFR.dat")
fd01_sfr=np.loadtxt(data_dir/"cloudfdp1_VSMDPL_z10_mhalo_SFR.dat")

assert base_sfr.shape==fd01_sfr.shape
assert np.allclose(base_sfr[:,0],fd01_sfr[:,0])

print("table shape:",base_sfr.shape)
print("mass range:",base_sfr[0,0],base_sfr[-1,0])

In [ ]:
lcdm_data=cv.load_data(
    base_path=cv.lcdm_base_path,
    snap_num=cv.snap_num,
)

lcdm_positions_cmpc,lcdm_MUV,lcdm_subhalo_mass,lcdm_valid=cv.select_valid_uv_galaxies(
    data=lcdm_data,
    run_type="LCDM",
)

In [ ]:
lcdm_field_lengths_cmpc=cv.calculate_rectangle_field_lengths(
    cosmology=cv.lumina_cosmology,
    z_center=cv.z_center,
    z_min=cv.z_min,
    z_max=cv.z_max,
    width_angle=cv.nircam_width,
    height_angle=cv.nircam_height,
)

lcdm_field_volume=np.prod(lcdm_field_lengths_cmpc)*u.Mpc**3

lcdm_field_ids,lcdm_inside,_,lcdm_n_fields=cv.assign_subhalos_to_fields(
    positions_cmpc=lcdm_positions_cmpc,
    box_size_cmpc=lcdm_data["box_size_cmpc"],
    field_volume=lcdm_field_volume,
    geometry="rectangle",
    field_lengths_cmpc=lcdm_field_lengths_cmpc,
)

lcdm_MUV_inside=lcdm_MUV[lcdm_inside]
lcdm_subhalo_mass_inside=lcdm_subhalo_mass[lcdm_inside]

assert len(lcdm_MUV_inside)==len(lcdm_subhalo_mass_inside)
assert len(lcdm_MUV_inside)==len(lcdm_field_ids)

print("field dimensions:",lcdm_field_lengths_cmpc)
print("field volume:",lcdm_field_volume)
print("number of fields:",lcdm_n_fields)
print("subhalos inside covered volume:",len(lcdm_MUV_inside))

In [ ]:
somerville_log_mhalo=base_sfr[:,0]
somerville_delta_log_sfr=fd01_sfr[:,2]-base_sfr[:,2]
somerville_sfr_boost=10**somerville_delta_log_sfr
somerville_delta_MUV=-2.5*somerville_delta_log_sfr

print("SFR boost range:",np.min(somerville_sfr_boost),np.max(somerville_sfr_boost))
print("MUV shift range:",np.min(somerville_delta_MUV),np.max(somerville_delta_MUV))

In [ ]:
lcdm_subhalo_mass_msun_inside=lcdm_subhalo_mass_inside/lcdm_data["h"]

lcdm_log_subhalo_mass_inside=np.full(len(lcdm_subhalo_mass_msun_inside),np.nan)

positive_mass=(
    np.isfinite(lcdm_subhalo_mass_msun_inside)
    &(lcdm_subhalo_mass_msun_inside>0)
)

lcdm_log_subhalo_mass_inside[positive_mass]=np.log10(
    lcdm_subhalo_mass_msun_inside[positive_mass]
)

somerville_valid=(
    np.isfinite(lcdm_MUV_inside)
    &np.isfinite(lcdm_log_subhalo_mass_inside)
    &(lcdm_log_subhalo_mass_inside>=somerville_log_mhalo[0])
    &(lcdm_log_subhalo_mass_inside<=somerville_log_mhalo[-1])
)

lcdm_MUV_somerville=lcdm_MUV_inside.copy()

lcdm_MUV_somerville[somerville_valid]=(
    lcdm_MUV_inside[somerville_valid]
    +np.interp(
        lcdm_log_subhalo_mass_inside[somerville_valid],
        somerville_log_mhalo,
        somerville_delta_MUV,
    )
)

assert len(lcdm_MUV_somerville)==len(lcdm_MUV_inside)

assert np.allclose(
    lcdm_MUV_somerville[~somerville_valid],
    lcdm_MUV_inside[~somerville_valid],
    equal_nan=True,
)

print("galaxies receiving boost:",np.count_nonzero(somerville_valid))
print("fraction receiving boost:",np.mean(somerville_valid))

In [ ]:
for MUV_limit in cv.MUV_limits:
    og_number=np.count_nonzero(lcdm_MUV_inside<MUV_limit)
    boosted_number=np.count_nonzero(lcdm_MUV_somerville<MUV_limit)
    boosted_outside=np.count_nonzero(
        (lcdm_MUV_somerville<MUV_limit)
        &~somerville_valid
    )

    print("MUV limit:",MUV_limit)
    print("og count:",og_number)
    print("boosted count:",boosted_number)
    print("boosted sample outside table:",boosted_outside)

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(13,5),facecolor="white")

ax[0].plot(
    base_sfr[:,0],
    base_sfr[:,2],
    color="black",
    marker="o",
    label="Baseline Kennicutt",
)

ax[0].fill_between(
    base_sfr[:,0],
    base_sfr[:,1],
    base_sfr[:,3],
    color="gray",
    alpha=0.25,
)

ax[0].plot(
    fd01_sfr[:,0],
    fd01_sfr[:,2],
    color="purple",
    marker="o",
    label=r"$f_{\rm dense}=0.1$",
)

ax[0].fill_between(
    fd01_sfr[:,0],
    fd01_sfr[:,1],
    fd01_sfr[:,3],
    color="purple",
    alpha=0.2,
)

ax[0].set_xlabel(r"$\log_{10}(M_{\rm halo}/M_\odot)$")
ax[0].set_ylabel(r"$\log_{10}(\mathrm{SFR}/M_\odot\,{\rm yr}^{-1})$")
ax[0].grid(alpha=0.3)
ax[0].legend()

ax[1].plot(
    somerville_log_mhalo,
    somerville_delta_MUV,
    color="purple",
    marker="o",
)

ax[1].axhline(0,color="black",linestyle="--")
ax[1].set_xlabel(r"$\log_{10}(M_{\rm halo}/M_\odot)$")
ax[1].set_ylabel(r"$\Delta M_{\rm UV}$")
ax[1].invert_yaxis()
ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
og_MUV_bins,og_log_phi=cv.compute_UVLF(
    MUV_values=lcdm_MUV_inside,
    field_volume=lcdm_field_volume,
    n_fields=lcdm_n_fields,
    MUV_bin_edges=cv.MUV_bin_edges,
    MUV_bin_width=cv.MUV_bin_width,
)

somerville_MUV_bins,somerville_log_phi=cv.compute_UVLF(
    MUV_values=lcdm_MUV_somerville,
    field_volume=lcdm_field_volume,
    n_fields=lcdm_n_fields,
    MUV_bin_edges=cv.MUV_bin_edges,
    MUV_bin_width=cv.MUV_bin_width,
)

fig,ax=plt.subplots(figsize=(9,7),facecolor="white")

ax.plot(
    og_MUV_bins,
    og_log_phi,
    marker="s",
    linestyle="-",
    color="gray",
    label="LUMINA",
)

ax.plot(
    somerville_MUV_bins,
    somerville_log_phi,
    marker="P",
    linestyle="-",
    color="purple",
    label=r"LUMINA+$f_{\rm dense}=0.1$",
)

cv.obs.plot_photoz_constraints(
    redshift=10,
    ax=ax,
    marker="o",
    ms=5,
    linestyle="none",
    color="black",
    capsize=4,
)

cv.obs.plot_specz_constraints(
    redshift=10,
    ax=ax,
    marker="s",
    ms=5,
    linestyle="none",
    color="red",
    capsize=4,
)

ax.set_xlabel(r"$M_{\rm UV}$")
ax.set_ylabel(r"$\log_{10}(\Phi/{\rm Mpc}^{-3}\,{\rm mag}^{-1})$")
ax.text(
    0.9,
    0.9,
    r"$z\simeq10$",
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=20,
)

ax.set_xlim(-24,-16)
ax.set_ylim(-7.5,0.5)
ax.grid(alpha=0.3)
ax.legend(loc="upper left",fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
cv.which_cosmology("LCDM")

lcdm_og_results=cv.og_UV_model(
    MUV_inside=lcdm_MUV_inside,
    subhalo_mass_inside=lcdm_subhalo_mass_inside,
    redshift=lcdm_data["redshift"],
    field_ids=lcdm_field_ids,
    n_fields=lcdm_n_fields,
    MUV_limits=cv.MUV_limits,
    model_name="LUMINA og",
)

lcdm_somerville_results=cv.og_UV_model(
    MUV_inside=lcdm_MUV_somerville,
    subhalo_mass_inside=lcdm_subhalo_mass_inside,
    redshift=lcdm_data["redshift"],
    field_ids=lcdm_field_ids,
    n_fields=lcdm_n_fields,
    MUV_limits=cv.MUV_limits,
    model_name="LUMINA Somerville fdense=0.1",
)

lcdm_A1_results=cv.A1_delta_MUV(MUV_inside=lcdm_MUV_inside,subhalo_mass_inside=lcdm_subhalo_mass_inside,redshift=lcdm_data["redshift"],field_ids=lcdm_field_ids,n_fields=lcdm_n_fields,MUV_limits=cv.MUV_limits,delta_MUV_list=[cv.best_delta_MUV])

lcdm_C1_results=cv.C1_sigma_UV(MUV_inside=lcdm_MUV_inside,subhalo_mass_inside=lcdm_subhalo_mass_inside,redshift=lcdm_data["redshift"],field_ids=lcdm_field_ids,n_fields=lcdm_n_fields,MUV_limits=cv.MUV_limits,sigma_UV_list=[cv.best_sigma_UV],seed=cv.random_seed)

lcdm_B2_results=cv.og_UV_model(MUV_inside=lcdm_MUV_B2_inside,subhalo_mass_inside=lcdm_subhalo_mass_inside,redshift=lcdm_data["redshift"],field_ids=lcdm_field_ids,n_fields=lcdm_n_fields,MUV_limits=cv.MUV_limits,model_name="B2")

lcdm_C2_results=cv.og_UV_model(MUV_inside=lcdm_MUV_C2,subhalo_mass_inside=lcdm_subhalo_mass_inside,redshift=lcdm_data["redshift"],field_ids=lcdm_field_ids,n_fields=lcdm_n_fields,MUV_limits=cv.MUV_limits,model_name="C2")

lcdm_shen_extrapolated_results=cv.og_UV_model(MUV_inside=lcdm_MUV_luminosity_extrapolated_all,subhalo_mass_inside=lcdm_subhalo_mass_inside,redshift=lcdm_data["redshift"],field_ids=lcdm_field_ids,n_fields=lcdm_n_fields,MUV_limits=cv.MUV_limits,model_name="Shen extrapolated")

In [ ]:
og_sigma_cv=np.array([
    result["sigma_cv"]
    for result in lcdm_og_results
])

og_error_low=np.array([
    result["sigma_cv_err_low"]
    for result in lcdm_og_results
])

og_error_high=np.array([
    result["sigma_cv_err_high"]
    for result in lcdm_og_results
])

somerville_sigma_cv=np.array([
    result["sigma_cv"]
    for result in lcdm_somerville_results
])

somerville_error_low=np.array([
    result["sigma_cv_err_low"]
    for result in lcdm_somerville_results
])

somerville_error_high=np.array([
    result["sigma_cv_err_high"]
    for result in lcdm_somerville_results
])

fig,ax=plt.subplots(figsize=(8,6),facecolor="white")

ax.errorbar(
    cv.MUV_limits,
    og_sigma_cv,
    yerr=[og_error_low,og_error_high],
    marker="s",
    linestyle="-",
    color="gray",
    capsize=4,
    label="LUMINA",
)

ax.errorbar(
    cv.MUV_limits,
    somerville_sigma_cv,
    yerr=[somerville_error_low,somerville_error_high],
    marker="P",
    linestyle="-",
    color="purple",
    capsize=4,
    label=r"LUMINA+$f_{\rm dense}=0.1$",
)

ax.scatter(
    cv.MUV_limits,
    cv.obs_sigma_cv,
    marker="*",
    s=220,
    color="red",
    label="PANORAMIC obs",
    zorder=10,
)

ax.errorbar(
    cv.MUV_limits,
    cv.obs_sigma_cv,
    yerr=[cv.obs_lower_error,cv.obs_upper_error],
    ecolor="red",
    fmt="none",
    capsize=6,
)

ax.set_xlabel(r"$M_{\rm UV}$ limit")
ax.set_ylabel(r"$\sigma_{\rm CV}$")
ax.text(
    0.9,
    0.9,
    r"$z\simeq10$",
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=20,
)
ax.set_xlim(-19.3,-20.7)
ax.grid(alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
lcdm_sigma_m=cv.compute_sigma_m(
    cosmology=cv.which_cosmology("LCDM"),
    field_lengths_cmpc=lcdm_field_lengths_cmpc,
    box_size_cmpc=lcdm_data["box_size_cmpc"],
    h=lcdm_data["h"],
    redshift=lcdm_data["redshift"],
)

print("sigma_m:",lcdm_sigma_m)

In [ ]:
for results in [lcdm_og_results,lcdm_somerville_results]:
    for result in results:
        result["sigma_m"]=lcdm_sigma_m
        result["sigma_cv_predicted"]=result["b_eff"]*lcdm_sigma_m
        result["cv_ratio"]=result["sigma_cv"]/result["sigma_cv_predicted"]

print(
    "model",
    "MUV_limit",
    "measured_cv",
    "predicted_cv",
    "ratio",
)

for results in [lcdm_og_results,lcdm_somerville_results]:
    for result in results:
        print(
            result["model"],
            result["MUV_limit"],
            result["sigma_cv"],
            result["sigma_cv_predicted"],
            result["cv_ratio"],
        )

In [ ]:
og_ratio=np.array([
    result["cv_ratio"]
    for result in lcdm_og_results
])

og_predicted=np.array([
    result["sigma_cv_predicted"]
    for result in lcdm_og_results
])

og_ratio_error_low=og_error_low/og_predicted
og_ratio_error_high=og_error_high/og_predicted

somerville_ratio=np.array([
    result["cv_ratio"]
    for result in lcdm_somerville_results
])

somerville_predicted=np.array([
    result["sigma_cv_predicted"]
    for result in lcdm_somerville_results
])

somerville_ratio_error_low=somerville_error_low/somerville_predicted
somerville_ratio_error_high=somerville_error_high/somerville_predicted

fig,ax=plt.subplots(figsize=(8,6),facecolor="white")

ax.errorbar(
    cv.MUV_limits,
    og_ratio,
    yerr=[og_ratio_error_low,og_ratio_error_high],
    marker="s",
    linestyle="-",
    color="gray",
    capsize=4,
    label="LUMINA",
)

ax.errorbar(
    cv.MUV_limits,
    somerville_ratio,
    yerr=[somerville_ratio_error_low,somerville_ratio_error_high],
    marker="P",
    linestyle="-",
    color="purple",
    capsize=4,
    label=r"LUMINA+$f_{\rm dense}=0.1$",
)

ax.axhline(1,color="black",linestyle="--")

ax.set_xlabel(r"$M_{\rm UV}$ limit")
ax.set_ylabel(
    r"$\sigma_{\rm CV}^{\rm measured}/"
    r"(b_{\rm eff}\sigma_m)$"
)
ax.text(
    0.9,
    0.9,
    r"$z\simeq10$",
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=20,
)
ax.set_xlim(-19.3,-20.7)
ax.grid(alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
all_series=[
    (lcdm_og_results,"LUMINA","s","gray"),
    (lcdm_A1_results,"A1","D","tab:blue"),
    (lcdm_C1_results,"C1","o","tab:orange"),
    (lcdm_B2_results,"B2","P","tab:purple"),
    (lcdm_C2_results,"C2","X","tab:brown"),
    (lcdm_somerville_results,r"Somerville $f_{\rm dense}=0.1$","v","tab:cyan"),
    (lcdm_shen_capped_results,"Shen capped",">","navy"),
    (lcdm_shen_extrapolated_results,"Shen extrapolated","<","magenta"),
    (ede_results,"EDE","^","tab:green"),
]

fig,ax=plt.subplots(1,3,figsize=(20,5),facecolor="white")

for results,label,marker,color in all_series:
    limits=np.array([result["MUV_limit"] for result in results])
    mu=np.array([result["mu"] for result in results])
    sigma_cv=np.array([result["sigma_cv"] for result in results])
    sigma_cv_low=np.array([result["sigma_cv_err_low"] for result in results])
    sigma_cv_high=np.array([result["sigma_cv_err_high"] for result in results])
    predicted=np.array([result["sigma_cv_predicted"] for result in results])
    ratio=sigma_cv/predicted

    ax[0].plot(limits,mu,marker=marker,color=color,linestyle="-",label=label)
    ax[1].errorbar(limits,sigma_cv,yerr=[sigma_cv_low,sigma_cv_high],marker=marker,color=color,linestyle="-",capsize=3,label=label)
    ax[2].errorbar(limits,ratio,yerr=[sigma_cv_low/predicted,sigma_cv_high/predicted],marker=marker,color=color,linestyle="-",capsize=3,label=label)

ax[0].scatter(cv.obs_MUV_limits,cv.obs_mu,marker="*",s=180,color="red",zorder=10)
ax[1].errorbar(cv.obs_MUV_limits,cv.obs_sigma_cv,yerr=[cv.obs_lower_error,cv.obs_upper_error],marker="*",markersize=14,color="red",linestyle="none",capsize=4,zorder=10)
ax[2].axhline(1,color="black",linestyle="--")

ax[0].set_ylabel(r"$\mu$")
ax[1].set_ylabel(r"$\sigma_{\rm CV}^{\rm measured}$")
ax[2].set_ylabel(r"$\sigma_{\rm CV}^{\rm measured}/(b_{\rm eff}\sigma_m)$")

for axis in ax:
    axis.set_xlabel(r"$M_{\rm UV}$ limit")
    axis.set_xlim(-19.3,-20.7)
    axis.grid(alpha=0.3)

ax[2].legend(fontsize=9,bbox_to_anchor=(1.02,1),loc="upper left")
plt.tight_layout()
plt.show()